In [1]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

PROJECT_ROOT = Path("..")

# Use the combined all-farms features file (218k windows vs 49k for Farm A only)
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "all_farms_features.parquet"

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE    = 0.20


In [2]:
df = pd.read_parquet(DATA_PATH)

print(f"Raw dataset shape : {df.shape}")
print(f"Farms             : {sorted(df['farm'].unique())}")
print(f"Events per farm   :")
for farm, grp in df.groupby("farm"):
    print(f"  Farm {farm}: {grp['event_id'].nunique()} events, "
          f"{len(grp):,} windows  (normal={int((grp['target']==0).sum())}, anomaly={int((grp['target']==1).sum())})")

# Encode farm as a numeric feature (A=0, B=1, C=2) before splitting
df["farm_enc"] = df["farm"].map({"A": 0, "B": 1, "C": 2})

META_COLS = ["event_id", "label", "target", "farm"]
X_raw = df.drop(columns=META_COLS)
y      = df["target"]

# Create a composite group key so event IDs are unique across farms
# (e.g. Farm A event 10 and Farm B event 10 would otherwise collide)
groups = df["farm"] + "_" + df["event_id"].astype(str)

# -----------------------------------------------------------------
# Remove constant / near-zero-variance columns (cross-farm zero pads)
# -----------------------------------------------------------------
vt = VarianceThreshold(threshold=0.0)   # drop columns with zero variance
X  = pd.DataFrame(
    vt.fit_transform(X_raw),
    columns=X_raw.columns[vt.get_support()],
    index=X_raw.index,
)

print(f"\nFeatures after removing zero-variance columns : {X.shape[1]}  (dropped {X_raw.shape[1] - X.shape[1]})")
print(f"Samples total : {len(X):,}")
print(f"\nClass distribution:")
print(y.value_counts().rename({0: "Normal", 1: "Anomaly"}))


Raw dataset shape : (218414, 1020)
Farms             : ['A', 'B', 'C']
Events per farm   :
  Farm A: 22 events, 49,851 windows  (normal=25051, anomaly=24800)
  Farm B: 15 events, 35,784 windows  (normal=21065, anomaly=14719)
  Farm C: 58 events, 132,779 windows  (normal=70783, anomaly=61996)

Features after removing zero-variance columns : 1017  (dropped 0)
Samples total : 218,414

Class distribution:
target
Normal     116899
Anomaly    101515
Name: count, dtype: int64


In [3]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train_raw = X.iloc[train_idx].copy()
X_test_raw  = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx]
y_test  = y.iloc[test_idx]

train_groups = groups.iloc[train_idx]
test_groups  = groups.iloc[test_idx]

print(f"Training   : {len(X_train_raw):,} windows | {train_groups.nunique()} events")
print(f"Testing    : {len(X_test_raw):,} windows  | {test_groups.nunique()} events")
print(f"\nTrain class dist: {y_train.value_counts().to_dict()}")
print(f"Test  class dist: {y_test.value_counts().to_dict()}")
print(f"\nFarm split:")
farm_col_train = df["farm"].iloc[train_idx]
farm_col_test  = df["farm"].iloc[test_idx]
for farm in ["A","B","C"]:
    tr = (farm_col_train == farm).sum()
    te = (farm_col_test  == farm).sum()
    print(f"  Farm {farm}: {tr:,} train | {te:,} test windows")


Training   : 174,858 windows | 76 events
Testing    : 43,556 windows  | 19 events

Train class dist: {0: 96320, 1: 78538}
Test  class dist: {1: 22977, 0: 20579}

Farm split:
  Farm A: 38,483 train | 11,368 test windows
  Farm B: 28,544 train | 7,240 test windows
  Farm C: 107,831 train | 24,948 test windows


In [4]:
# ─────────────────────────────────────────────────────────────
# Stage 1 – Isolation Forest (unsupervised anomaly scorer)
#   Fit ONLY on normal training windows so "normal" is the
#   learned reference. The score_samples() output becomes an
#   extra feature for the downstream classifier.
# ─────────────────────────────────────────────────────────────
normal_train_mask = (y_train == 0)

iso = IsolationForest(
    n_estimators=200,
    contamination=0.05,   # expected fraction of anomalies in "normal" data
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
iso.fit(X_train_raw[normal_train_mask])

# score_samples: more-negative → more anomalous
X_train = X_train_raw.copy()
X_train["iso_anomaly_score"] = iso.score_samples(X_train_raw)

X_test = X_test_raw.copy()
X_test["iso_anomaly_score"] = iso.score_samples(X_test_raw)

print(f"Isolation Forest fitted on {normal_train_mask.sum():,} normal training windows.")
print(f"Feature set size now: {X_train.shape[1]}  (added iso_anomaly_score)")


Isolation Forest fitted on 96,320 normal training windows.
Feature set size now: 1018  (added iso_anomaly_score)


In [5]:
# ─────────────────────────────────────────────────────────────
# Stage 2 – Random Forest classifier
#   class_weight='balanced' guards against future imbalance;
#   it has little effect now since classes are nearly 1:1.
# ─────────────────────────────────────────────────────────────
model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model.fit(X_train, y_train)
print("Training complete.")


Training complete.


In [6]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]   # probability of anomaly class


In [7]:
# ─────────────────────────────────────────────────────────────
# Metrics  (PR-AUC is the primary metric for fault detection;
#           ROC-AUC and per-class recall are secondary)
# ─────────────────────────────────────────────────────────────
pr_auc   = average_precision_score(y_test, y_prob)
roc_auc  = roc_auc_score(y_test, y_prob)
anomaly_recall    = recall_score(y_test, y_pred)
anomaly_precision = precision_score(y_test, y_pred)
anomaly_f1        = f1_score(y_test, y_pred)

results = pd.DataFrame({
    "Metric": [
        "PR-AUC (primary)",
        "ROC-AUC",
        "Anomaly Recall",
        "Anomaly Precision",
        "Anomaly F1",
        "Accuracy",
    ],
    "Value": [
        pr_auc,
        roc_auc,
        anomaly_recall,
        anomaly_precision,
        anomaly_f1,
        accuracy_score(y_test, y_pred),
    ],
}).assign(Value=lambda d: d["Value"].round(4))

display(results)


,Metric,Value
0,PR-AUC (primary),0.3660
1,ROC-AUC,0.1900
2,Anomaly Recall,0.4421
3,Anomaly Precision,0.3756
4,Anomaly F1,0.4061
5,Accuracy,0.3179


In [8]:
print(classification_report(y_test, y_pred, target_names=["normal", "anomaly"]))


              precision    recall  f1-score   support

      normal       0.22      0.18      0.20     20579
     anomaly       0.38      0.44      0.41     22977

    accuracy                           0.32     43556
   macro avg       0.30      0.31      0.30     43556
weighted avg       0.30      0.32      0.31     43556



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Raw confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["normal", "anomaly"],
    ax=axes[0],
)
axes[0].set_title("Confusion Matrix (counts)")

# Normalized confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=["normal", "anomaly"],
    normalize="true",
    ax=axes[1],
)
axes[1].set_title("Confusion Matrix (row-normalized)")

plt.tight_layout()
plt.show()


Training Performance
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     20461
           1       1.00      1.00      1.00     18006

    accuracy                           1.00     38467
   macro avg       1.00      1.00      1.00     38467
weighted avg       1.00      1.00      1.00     38467



In [ ]:
# Check training-set performance to detect over/underfitting
train_pred = model.predict(X_train)
print("Training-set performance:")
print(classification_report(y_train, train_pred, target_names=["normal", "anomaly"]))


,Feature,Importance
164,sensor_41_avg_mean,0.016225
166,sensor_41_avg_min,0.011353
167,sensor_41_avg_max,0.011276
132,sensor_33_avg_mean,0.011173
128,sensor_32_avg_mean,0.010658
160,sensor_40_avg_mean,0.009600
136,sensor_34_avg_mean,0.009529
152,sensor_38_avg_mean,0.009194
36,sensor_9_avg_mean,0.008380
137,sensor_34_avg_std,0.008371


In [ ]:
importance = (
    pd.DataFrame({
        "Feature":    X_train.columns,
        "Importance": model.feature_importances_,
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

# Show top 20 features
print("Top 20 most important features:")
display(importance.head(20))

# Quick bar chart
fig, ax = plt.subplots(figsize=(10, 5))
importance.head(20).plot.barh(
    x="Feature", y="Importance", ax=ax, legend=False
)
ax.invert_yaxis()
ax.set_title("Feature Importance — Top 20")
ax.set_xlabel("Mean decrease in impurity")
plt.tight_layout()
plt.show()


Model saved.


In [ ]:
# Save Isolation Forest, classifier, and the column list used at training time
joblib.dump(iso,   MODEL_DIR / "isolation_forest.pkl")
joblib.dump(model, MODEL_DIR / "random_forest.pkl")
joblib.dump(list(X_train_raw.columns), MODEL_DIR / "feature_names.pkl")
joblib.dump(vt,    MODEL_DIR / "variance_threshold.pkl")

print("Saved:")
print(f"  {MODEL_DIR / 'isolation_forest.pkl'}")
print(f"  {MODEL_DIR / 'random_forest.pkl'}")
print(f"  {MODEL_DIR / 'feature_names.pkl'}   ({len(X_train_raw.columns)} features)")
print(f"  {MODEL_DIR / 'variance_threshold.pkl'}")

print()
print("=" * 55)
print("Training Summary")
print("=" * 55)
print(f"Dataset          : all_farms_features.parquet")
print(f"Total samples    : {len(df):,}")
print(f"Features (raw)   : {X_raw.shape[1]}")
print(f"Features (used)  : {X_train.shape[1]}  (excl. zero-variance)")
print(f"Events (total)   : {groups.nunique()}")
print(f"Train / test     : {len(X_train):,} / {len(X_test):,} windows")
print()
print(results.to_string(index=False))


Training Summary
Samples       : 49,851
Features      : 1016
Events        : 22

      Metric     Value
0   Accuracy  0.344431
1  Precision  0.404836
2     Recall  0.209450
3   F1 Score  0.276069
4    ROC AUC  0.496465
